Partie 1 – Exploration du dataset
Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe,
son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa
taille.
NB : prendre en charge aussi les fichiers corrompus

In [ ]:
#1. Importer les bibliothèques
# Bibliothèque pour manipuler les chemins et les dossiers
from pathlib import Path

# Bibliothèque pour lire et vérifier les images
from PIL import Image, UnidentifiedImageError

# Bibliothèque pour effectuer des calculs numériques
import numpy as np

# Bibliothèque pour créer et manipuler des tableaux de données
import pandas as pd

#2. definir le chemin du dataset
# Chemin vers le dossier contenant les images originales
RAW_DIR = Path("../data/raw")


#3. Fonction pour analyser une image
def analyser_image(image_path):
    """
    Récupère les informations demandées pour une image.
    Les images corrompues sont prises en charge.
    """

    # Informations par défaut
    informations = {
        "nom": image_path.name,
        "classe": image_path.parent.name,
        "format": None,
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nombre_canaux": None,
        "taille": image_path.stat().st_size,
        "corrompue": False
    }

    try:
        # Ouverture de l'image
        with Image.open(image_path) as image:

            # Conversion de l'image en tableau NumPy
            pixels = np.array(image)

            # Récupération du format
            informations["format"] = image.format

            # Récupération du mode
            informations["mode"] = image.mode

            # Récupération de la largeur
            informations["largeur"] = image.width

            # Récupération de la hauteur
            informations["hauteur"] = image.height

            # Calcul de l'écart-type des pixels
            informations["ecart_type_pixels"] = pixels.std()

            # Récupération du nombre de canaux
            informations["nombre_canaux"] = len(image.getbands())

    except (UnidentifiedImageError, OSError, ValueError):

        # Si l'image ne peut pas être lue,
        # elle est considérée comme corrompue
        informations["corrompue"] = True

    return informations


In [9]:
#4. Parcourir toutes les images du dataset
# Liste qui contiendra les informations de toutes les images
resultats = []

# Parcours des dossiers de classes
for classe_dir in RAW_DIR.iterdir():

    # Vérifie qu'il s'agit bien d'un dossier
    if classe_dir.is_dir():

        # Parcours des fichiers du dossier
        for image_path in classe_dir.iterdir():

            # Vérifie qu'il s'agit bien d'un fichier
            if image_path.is_file():

                # Analyse de l'image
                informations = analyser_image(image_path)

                # Ajout des informations à la liste
                resultats.append(informations)

In [6]:
#5. Créer le tableau d'exploration
# Transformation de la liste en DataFrame
df_images = pd.DataFrame(resultats)

# Affichage des premières lignes
df_images.head()

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille,corrompue
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False


In [8]:
# Afficher le DataFrame complet
df_images

,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nombre_canaux,taille,corrompue
0,cardboard1.jpg,cardboard,JPEG,RGB,512.0,384.0,40.588529,3.0,17333,False
1,cardboard10.jpg,cardboard,JPEG,RGB,512.0,384.0,42.571288,3.0,21683,False
2,cardboard100.jpg,cardboard,JPEG,RGB,512.0,384.0,46.108305,3.0,14884,False
3,cardboard101.jpg,cardboard,JPEG,RGB,512.0,384.0,72.263996,3.0,14289,False
4,cardboard102.jpg,cardboard,JPEG,RGB,512.0,384.0,48.388937,3.0,18015,False
...,...,...,...,...,...,...,...,...,...,...
1027,trash50.jpg,trash,JPEG,RGB,512.0,384.0,41.780027,3.0,13563,False
1028,trash6.jpg,trash,JPEG,RGB,512.0,384.0,37.391702,3.0,10534,False
1029,trash7.jpg,trash,JPEG,RGB,512.0,384.0,41.297376,3.0,8011,False
1030,trash8.jpg,trash,JPEG,RGB,512.0,384.0,47.635953,3.0,16946,False


Partie 2 – Détecter les images corrompues
Écrire et se servir d’une fonction qui détecte une image corrompue

In [10]:
#1. Fonction de détection
def image_corrompue(image_path):
    """
    Vérifie si une image est corrompue.

    Retourne :
    True  -> image corrompue
    False -> image valide
    """

    try:
        # Ouverture de l'image
        with Image.open(image_path) as image:

            # Vérification de l'intégrité du fichier
            image.verify()

        # Aucune erreur : l'image est valide
        return False

    except (UnidentifiedImageError, OSError, ValueError):

        # Une erreur signifie que l'image est corrompue
        return True